# Synthon-Based Library Design

This tutorial demonstrates the synthon subsystem of ``SynPlanner`` — a native port of the Synt-On / SynthI toolkit (Zabolotna *et al.*, *J. Chem. Inf. Model.* **2022**, 62, 2151-2163).

A **synthon** is a valence-complete fragment of a molecule whose atoms carry a label saying how that atom is going to react. ``SynPlanner`` writes that label into the SMILES, so `c1ccccc1[NH2_nuc]` is aniline acting as a nucleophile through its nitrogen. Because the label is a property of the graph and not of the string, a synthon can be cut, canonicalised, re-read and joined without ever losing track of which atom is the reaction centre.

The tutorial walks the same arc as the reference toolkit's own documentation, in the ``SynPlanner`` API:

1. classify a building block
2. turn it into synthons
3. scaffolds and the rule of two
4. cut a target into a disconnection DAG
5. score that DAG against a synthon stock
6. enumerate — a full combinatorial library, and analogues of one target
7. positional analogue scanning
8. ring closure, which the reference has no counterpart for

## Basic recommendations

1. Every molecule handed to the synthon code must be canonicalised first — ``chython`` does not aromatise on parse, and a kekulised input silently matches no SMARTS at all. The `*_smiles` entry points used below do this for you.

2. A target must go through `Fragmenter` (or `fragment_smiles`) to become a `SynthonContainer`. On a plain `MoleculeContainer` the labels do not serialise and everything downstream quietly sees an unlabelled molecule.

3. Note the namespace: `synthon_cgr` in the [clustering tutorial](07_Clustering.ipynb) means a *strategic bond* of a finished route. It is unrelated to the synthons here.

## 1. Set up input and output data locations

The synthon subsystem needs no downloaded model — the whole knowledge base (147 building-block classes over 2401 SMARTS, 147 synthonisation programs, 154 disconnection rules) ships inside the package.

The building blocks used below are the nine the reference toolkit ships with its own tutorial, so the numbers in this notebook are the published ones.

In [1]:
from pathlib import Path

from chython import depict_settings, grid_depict, synthon_smiles
from IPython.display import SVG, display

# a synthon draws its labels as E / Nu markers; atom-atom mapping numbers only clutter them
depict_settings(aam=False)


def show(structures, labels=None):
    """Draw molecules, or synthon SMILES, side by side."""
    molecules = []
    for item in structures:
        molecule = synthon_smiles(item) if isinstance(item, str) else item
        molecule.canonicalize()
        molecules.append(molecule)
    display(SVG(grid_depict(molecules, labels=labels)))


# the reference toolkit's own tutorial building blocks, shipped with the test fixtures
data_folder = next(
    p
    for p in (Path("../tests/data/synthon"), Path("tests/data/synthon"))
    if p.exists()
)

# results folder (shared with the other tutorials)
results_folder = Path("tutorial_results").resolve()
results_folder.mkdir(exist_ok=True)

building_blocks = [
    line.split("\t")
    for line in (data_folder / "BBs.cxsmiles").read_text().splitlines()
    if line.strip()
]
for smi, name in building_blocks:
    print(f"{smi:20s} {name}")

CCCCCC(=O)C          heptan-2-one
CCO                  ethan-1-ol
CCCCO                butan-1-ol
C1=CC=C(C=C1)CO      benzylic_alcohol
C1=CC=C(C=C1)N       aniline
C1=CC=C(C=C1)NC2=CC=CC=C2 diphenylamine
CCCCCC=O             hexanal
CCCCC(O)=O           pentanoic-acid
CCC(O)=O             propanoic-acid


## 2. Building-block classification

`BBClassifier` assigns a building block to one or more of 147 ordered classes. A class fires when *any* of its `at_least_one` patterns match, *all* of its `also` patterns match, and *none* of its exclusion patterns do — 93% of the shipped SMARTS are exclusions, which is why a class list is short and specific rather than a pile of near-misses.

The class order is load-bearing: the synthoniser breaks on the first poly-functional class and composes the mono-functional ones in file order.

In [2]:
from synplan.chem.synthon.classify import BBClassifier

classifier = BBClassifier()
print(f"{len(classifier.classes)} building-block classes loaded")

# the reference's own worked example: an amino-ester
classifier.classify_smiles("CCOC(=O)C1=C(N)SC=C1C2CC2")

147 building-block classes loaded


['Bifunctional_Amine_Ester', 'PrimaryAmines_PriAmines_Het-Anilines']

`classify_smiles` classifies every `.`-separated component on its own and unions the results back into class order. Matching the whole string instead would pair a group in one component against a group in another and invent a class, and one counter-ion's exclusion patterns would wipe out the parent's classes.

A building block whose reactivity the knowledge base does not cover comes back with an empty list, and an unparsable row comes back as `None`.

In [3]:
for smi in [
    "CCOC(=O)C1=C(N)SC=C1C2CC2",  # amino-ester
    "OC(=O)C(F)(F)F.CCCCCC(=O)C",  # ketone as its TFA salt — components classified apart
    "CCOC=1C=C(CC#N)C=CC1OCC(F)(F)F",  # a heterocyclisation reagent: no acyclic class
]:
    print(f"{smi}\n    {classifier.classify_smiles(smi)}")

CCOC(=O)C1=C(N)SC=C1C2CC2
    ['Bifunctional_Amine_Ester', 'PrimaryAmines_PriAmines_Het-Anilines']
OC(=O)C(F)(F)F.CCCCCC(=O)C
    ['Acid_Aliphatic_Acid', 'Ketones_Ketones']
CCOC=1C=C(CC#N)C=CC1OCC(F)(F)F
    []


For a whole catalogue, the `synplan bb_classifying` CLI command does the same thing over a file, in parallel.

## 3. Synthonisation

`BBSynthoniser` runs the rule program attached to each class and returns every synthon the building block can act as. The label records *how* the atom reacts, and it lives on the atom, not in the string.

`SynPlanner` uses eight labels where the paper's table lists nine — code 11 ("electrophilic nitrogen") collapses into `elec`, because the compatibility table has no `N:10` key to distinguish it from, so on nitrogen "electrophile" already has exactly one meaning.

| ``SynPlanner`` label | Paper code | Nature of the reaction centre | Example |
| --- | --- | --- | --- |
| `elec` | `:10`, `:11` | electrophilic | `C1CC1[CH3_elec]` |
| `nuc` | `:20` | nucleophilic | `C1CC1[NH2_nuc]` |
| `elecB` | `:21` | boronics-derived nucleophilic partner | `c1ccccc1[CH3_elecB]` |
| `elec2` | `:30` | bivalent electrophilic | `C1CC1C[CH3_elec2]` |
| `nuc2` | `:40` | bivalent nucleophilic | `C1CC1C[NH2_nuc2]` |
| `neut2` | `:50` | bivalent neutral (metathesis) | `C1CC1[CH3_neut2]` |
| `elec*` | `:60` | electrophilic radical | `c1cc[cH_elec*]nc1` |
| `nuc*` | `:70` | nucleophilic radical | `C1CC1[CH3_nuc*]` |

<div class="alert alert-info">
<b>Note</b>

Importing the synthoniser emits one `StereoDiscardedWarning`. It comes from canonicalising the module's own list of solvents to ignore (maleic and fumaric acid), not from your molecule.
</div>

In [4]:
from synplan.chem.synthon.synthonise import BBSynthoniser

synthoniser = BBSynthoniser(classifier=classifier)

produced = synthoniser.synthonise_smiles("CCOC(=O)C1=C(N)SC=C1C2CC2")
classes = sorted({name for record in produced.values() for name in record["classes"]})
print(f"{len(produced)} synthons from {'+'.join(classes)}")
show(produced)

4 synthons from Bifunctional_Amine_Ester


/Volumes/Lacie Tagir/Programming/Dev/synplan/wt/SynPlanner/synplan/chem/synthon/synthonise.py:16: StereoDiscardedWarning: Input stereochemistry is being discarded: SynPlanner's rule application and its synthon/building-block stock are keyed on flat structures, so any route proposed for this molecule is racemic / relative configuration not determined.
  str(safe_canonicalization(smiles(s)))


In [5]:
# the label is a graph property, so it survives the round trip through SMILES
for synthon in produced:
    molecule = synthon_smiles(synthon)
    molecule.canonicalize()
    print(f"{synthon:40s} {molecule.synthon_labels}")

C1CC1c2csc([NH2_nuc])c2C(=O)O            {8: 'nuc'}
C1CC1c2c(C(O)=O)c([NH2_nuc2])sc2         {10: 'nuc2'}
C1CC1c2csc([NH2_nuc])c2[CH_elec]=O       {8: 'nuc', 10: 'elec'}
C1CC1c2csc([NH2_nuc2])c2[CH_elec]=O      {8: 'nuc2', 10: 'elec'}


### Protecting groups

By default a protected intermediate is thrown away — the ethyl ester above appears only as the free acid and as the aldehyde it can be reduced to. `keep_protecting_groups` keeps both forms.

In [6]:
from synplan.chem.synthon.config import SynthonConfig

keep_pg = BBSynthoniser(SynthonConfig(keep_protecting_groups=True), classifier=classifier)

print("default  :", len(produced), "synthons")
print("keep PG  :", len(keep_pg.synthonise_smiles("CCOC(=O)C1=C(N)SC=C1C2CC2")), "synthons")

default  : 4 synthons
keep PG  : 6 synthons


### Salts and solvates

Real catalogues are full of TFA and hydrochloride salts. A building block should be desalted before it reaches the synthoniser, but the common counter-ions and solvents are dropped anyway: the trifluoroacetate below is ignored and only the ketone is cut, which the component index records.

In [7]:
salt = synthoniser.synthonise_smiles("OC(=O)C(F)(F)F.O=C1CCCC2(CCCC2)C1")
components = {record["component"] for record in salt.values()}

print(f"{len(salt)} synthons, all from component {components} — the ketone, not the trifluoroacetate")
show(salt)

6 synthons, all from component {1} — the ketone, not the trifluoroacetate


### Building the synthon stock

A **stock** is the whole catalogue synthonised once: `synthon SMILES -> the building blocks that produce it`. It is what makes a disconnection *available*, and what fills a slot during enumeration.

The nine reference building blocks give 18 synthons.

In [8]:
from synplan.chem.synthon.stock import load_synthon_stock
from synplan.interfaces.synthon_commands import synthonise_file

stock_path = results_folder / "tutorial_synthons.smi"
written, forced = synthonise_file(
    str(data_folder / "BBs.cxsmiles"), str(stock_path), SynthonConfig()
)

tutorial_stock = load_synthon_stock(str(stock_path))
print(f"{len(building_blocks)} building blocks -> {written} synthons; {forced} forced to keep protecting groups\n")
print(stock_path.read_text())

9 building blocks -> 18 synthons; 0 forced to keep protecting groups

C([CH2_elec]C)CCCC	CCCCCC(=O)C	Ketones_Ketones	0
CCCC[CH2_nuc2]C(=O)C	CCCCCC(=O)C	Ketones_Ketones	0
C(CCCC)[CH_elec](O)C	CCCCCC(=O)C	Ketones_Ketones	0
[CH_elec]([OH_nuc])(CCCCC)C	CCCCCC(=O)C	Alcohols_Aliphatic_alcohols+Ketones_Ketones	0
[OH_nuc]CC	CCO	Alcohols_Aliphatic_alcohols	0
[CH3_elec]C	CCO	Alcohols_Aliphatic_alcohols	0
[OH_nuc]CCCC	CCCCO	Alcohols_Aliphatic_alcohols	0
CCC[CH3_elec]	CCCCO	Alcohols_Aliphatic_alcohols	0
c1ccccc1C[OH_nuc]	C1=CC=C(C=C1)CO	Alcohols_Aliphatic_alcohols	0
c1cc(ccc1)[CH3_elec]	C1=CC=C(C=C1)CO	Alcohols_Aliphatic_alcohols	0
c1ccccc1[NH2_nuc]	C1=CC=C(C=C1)N	PrimaryAmines_PriAmines_Anilines	0
c1ccccc1[NH2_nuc2]	C1=CC=C(C=C1)N	PrimaryAmines_PriAmines_Anilines	0
c1ccccc1[NH_nuc]c2ccccc2	C1=CC=C(C=C1)NC2=CC=CC=C2	SecondaryAmines_diArylAmines	0
[CH3_elec2]CCCCC	CCCCCC=O	Aldehyde_Aliphatic_Aldehydes	0
C(C[CH3_elec])CCC	CCCCCC=O	Aldehyde_Aliphatic_Aldehydes	0
O[CH2_elec]CCCCC	CCCCCC=O	Aldehyde_Ali

`synthonise_file` is what the `synplan bb_synthonizing` command runs. Over a real catalogue it is the same call, fanned out over `num_workers` processes, and the stock file it writes is the currency of everything below.

## 4. Scaffolds and the rule of two

Two building-block analyses come with the port and are useful outside enumeration.

`scaffold_smiles` is a Bemis-Murcko scaffold taken *after* removing the ring-containing protecting and leaving groups — the rings that will not survive into the product and so say nothing about the building block. A molecule with no ring left returns `linearMolecule`.

In [9]:
from synplan.chem.scaffolds import scaffold_smiles

for smi, what in [
    ("O=C(O)c1ccc(NC(=O)OCC2c3ccccc3-c3ccccc32)cc1", "Fmoc-protected aniline"),
    ("CC1(C)OB(c2ccccc2)OC1(C)C", "pinacol boronate"),
    ("c1ccc(C2CO2)cc1", "styrene oxide"),
    ("CC(C)(C)OC(=O)N1CCNCC1", "Boc-piperazine"),
    ("CCO", "ethanol"),
]:
    print(f"{scaffold_smiles(smi):12s}  {what}")

c1ccccc1      Fmoc-protected aniline
c1ccccc1      pinacol boronate
c1ccccc1      styrene oxide
C1CNCCN1      Boc-piperazine
linearMolecule  ethanol


The **rule of two** is the reagent-likeness filter from the paper: MW <= 200, logP <= 2, H-bond donors <= 2, acceptors <= 4, kept in `RO2_LIMITS`. It answers "is this fragment small enough to be a reagent", which is a different question from whether it is in stock.

All four have to hold, and they fail independently: the spiro synthon below is a third under the weight limit and does no H-bonding at all, and is still rejected — on logP alone.

In [10]:
from rdkit.Chem import AddHs, Crippen
from rdkit.Chem.rdMolDescriptors import CalcExactMolWt, CalcNumHBA, CalcNumHBD

from synplan.chem.synthon.stock import RO2_LIMITS, ro2_pass

candidates = ["O=C(O)c1c(C2CC2)csc1[NH2_nuc]", "C1CCC2(C1)CCC[CH2_elec]C2"]
show(candidates, ["reagent-like", "too greasy"])

print(f"limits: MW <= {RO2_LIMITS[0]}, logP <= {RO2_LIMITS[1]}, HBD <= {RO2_LIMITS[2]}, HBA <= {RO2_LIMITS[3]}\n")
for smi in candidates:
    synthon = synthon_smiles(smi)
    synthon.canonicalize()
    plain = synthon.unlabelled().to_rdkit(keep_mapping=False)
    with_hydrogens = AddHs(plain)
    print(
        f"{smi:30s} MW={CalcExactMolWt(with_hydrogens):7.3f} logP={Crippen.MolLogP(plain):6.3f} "
        f"HBD={CalcNumHBD(with_hydrogens)} HBA={CalcNumHBA(with_hydrogens)} -> Ro2 {ro2_pass(synthon)}"
    )

limits: MW <= 200.0, logP <= 2.0, HBD <= 2, HBA <= 4

O=C(O)c1c(C2CC2)csc1[NH2_nuc]  MW=183.035 logP= 1.906 HBD=2 HBA=4 -> Ro2 True
C1CCC2(C1)CCC[CH2_elec]C2      MW=138.141 logP= 3.511 HBD=0 HBA=0 -> Ro2 False


`ro2_variant="paper"` (the default) reproduces the published Fig. 5 numbers; `"corrected"` applies the label-aware descriptor corrections the reference documents but never calls.

## 5. Fragmenting a target into a disconnection DAG

`fragment_smiles` cuts a target with the shipped disconnection rules, then cuts the pieces again, level by level, up to `max_stages`. The result is a `DisconnectionDAG`: every node is a **pathway** — a reagent *set*, with no step order, no intermediates and no yields — and an edge means "this pathway is one further cut of that one".

The target below is cenobamate, the reference documentation's own headline example. The reference cannot actually run it today: its labelling is `str.replace` on a SMILES string and stopped matching on RDKit 2023, so the example raises `TypeError`. The seven pathways its README documents are reproduced here rather than copied.

In [11]:
from synplan.chem.synthon.fragment import fragment_smiles

CENOBAMATE = "NC(=O)OC(CN1N=CN=N1)C1=CC=CC=C1Cl"

dag = fragment_smiles(CENOBAMATE)
print(f"target: {dag.target}")
print(f"{len(dag.pathways)} pathways\n")
for pathway in dag.best_available():
    print(f"{'|'.join(pathway.rules):28s} stage {pathway.depth}  {' . '.join(pathway.key)}")

# the top pathway drawn: E and Nu mark the atoms the cut bond hung on
top = dag.best_available()[0]
print(f"\ntop pathway, {'|'.join(top.rules)}:")
show(top.key)

target: n1n(ncn1)CC(c2c(cccc2)Cl)OC(N)=O
11 pathways

R2.2_0                       stage 1  [CH_elec](=O)N . c1(ccccc1C(Cn2nncn2)[OH_nuc])Cl
R5.1_0                       stage 1  [CH3_elec]C(OC(N)=O)c1ccccc1Cl . [nH]1cnn[n_nuc]1
R5.2_0                       stage 1  [nH]1cnn[n_nuc]1 . c1c(c(ccc1)C(OC(N)=O)[CH3_elecB])Cl
R2.2_0|R5.1_0                stage 2  [CH_elec](=O)N . [nH]1cnn[n_nuc]1 . c1cccc(c1Cl)C([OH_nuc])[CH3_elec]
R2.2_0|R5.2_0                stage 2  [CH_elec](=O)N . [OH_nuc]C(c1c(cccc1)Cl)[CH3_elecB] . [nH]1cnn[n_nuc]1
R2.2_0|R10.1_0               stage 2  [CH_elec](=O)N . c1c(Cl)c([CH2_elec][OH_nuc])ccc1 . n1(ncnn1)[CH3_nuc]
R2.2_0|R10.1_1               stage 2  [CH_elec](=O)N . [OH_nuc][CH2_elec]Cn1nncn1 . c1cccc(Cl)[cH_nuc]1
R5.1_0|R16.2b_0              stage 2  [CH2_elec]=[NH_nuc] . [CH3_elec]C(OC(N)=O)c1ccccc1Cl . [NH_elec]=N[NH2_nuc]
R5.2_0|R16.2b_0              stage 2  [CH2_elec]=[NH_nuc] . [NH_elec]=N[NH2_nuc] . c1c(c(ccc1)C(OC(N)=O)[CH3_elecB])Cl
R2.2_0|R5.1_0|R

The DAG is navigable. `roots()` are the one-cut pathways, `leaves()` the ones nothing cuts further, and `best_available()` sorts by availability first and then by the *fewest* reagents.

In [12]:
print("roots (one stage) :", len(dag.roots()))
print("leaves            :", len(dag.leaves()))
print("acyclic           :", dag.is_acyclic())

root = next(p for p in dag.roots() if p.rules == ("R2.2_0",))
print(f"\nchildren of {root.rules[0]}:")
for child in sorted(dag.children[root.key]):
    print("   ", " . ".join(child))

roots (one stage) : 3
leaves            : 4
acyclic           : True

children of R2.2_0:
    [CH_elec](=O)N . [OH_nuc]C(c1c(cccc1)Cl)[CH3_elecB] . [nH]1cnn[n_nuc]1
    [CH_elec](=O)N . [OH_nuc][CH2_elec]Cn1nncn1 . c1cccc(Cl)[cH_nuc]1
    [CH_elec](=O)N . [nH]1cnn[n_nuc]1 . c1cccc(c1Cl)C([OH_nuc])[CH3_elec]
    [CH_elec](=O)N . c1c(Cl)c([CH2_elec][OH_nuc])ccc1 . n1(ncnn1)[CH3_nuc]


### Choosing which rules to cut with

`rule_mode` and `rules_selection` pick the rule list. A selector is `Rn`, `Rn.m`, `Rn.ma`, a range `Rn-Rm`, or a comma-separated list of those; a bare `Rn` covers the whole family. A range is a slice of the *ordered* rule list, so `R1.2-R1.4` excludes `R1.1`. A selector that matches nothing is an error, never a silently empty run.

In [13]:
from synplan.chem.synthon.fragment import Fragmenter
from synplan.chem.utils import safe_canonicalization
from chython import smiles

target = safe_canonicalization(smiles(CENOBAMATE))

for mode, selection in [
    ("use_all", "R1-R13"),
    ("include_only", "R1-R9"),
    ("exclude_some", "R5.1"),
    ("one_by_one", "R2,R10,R5"),
]:
    config = SynthonConfig(rule_mode=mode, rules_selection=selection)
    found = Fragmenter(config).fragment(target)
    print(f"{mode:14s} {selection:10s} -> {len(found.pathways):2d} pathways")
    for pathway in found.best_available():
        print("      ", "|".join(pathway.rules))

use_all        R1-R13     -> 11 pathways
       R2.2_0
       R5.1_0
       R5.2_0
       R2.2_0|R5.1_0
       R2.2_0|R5.2_0
       R2.2_0|R10.1_0
       R2.2_0|R10.1_1
       R5.1_0|R16.2b_0
       R5.2_0|R16.2b_0
       R2.2_0|R5.1_0|R16.2b_0
       R2.2_0|R5.2_0|R16.2b_0
include_only   R1-R9      ->  5 pathways
       R2.2_0
       R5.1_0
       R5.2_0
       R2.2_0|R5.1_0
       R2.2_0|R5.2_0
exclude_some   R5.1       ->  7 pathways
       R2.2_0
       R5.2_0
       R2.2_0|R5.2_0
       R2.2_0|R10.1_0
       R2.2_0|R10.1_1
       R5.2_0|R16.2b_0
       R2.2_0|R5.2_0|R16.2b_0
one_by_one     R2,R10,R5  ->  2 pathways
       R2.2_0
       R2.2_0|R5.1_0


One rule of the published set ships here as two. The reference's `R12.3` carries two alternative labellings — Heck and Suzuki — but its loop overwrites its own result with no accumulator, so only the last one survives and the Heck labelling is dead code. Both ship, as `R12.3a` and `R12.3b`, which makes the acyclic rule count 39 rather than 38.

In [14]:
from synplan.chem.synthon.config import load_data

rules = load_data(SynthonConfig().rules_path)["disconnections"]
acyclic = [r for r in rules if not r["macro"] and not r["ring"]]
print(f"{len(rules)} rules shipped; {len(acyclic)} acyclic disconnections\n")

for rule in rules:
    if rule["id"].startswith("R12.3"):
        print(f"{rule['id']:8s} {rule['name']}")
        print(f"         {rule['smarts']}")

154 rules shipped; 39 acyclic disconnections

R12.3a   Heck and Suzuki coupling C(Ar) - C(sp2)
         [c:1]-!@[#6;X3;$([#6](=[#6])[#6,#1]):2]>>[c_elec:1].[C_nuc:2]
R12.3b   Heck and Suzuki coupling C(Ar) - C(sp2)
         [c:1]-!@[#6;X3;$([#6](=[#6])[#6,#1]):2]>>[c_nuc:1].[C_elec:2]


## 6. Availability against a synthon stock

Pass a stock to `Fragmenter` and every pathway gets an **availability rate**: the fraction of the target's atoms that come from synthons the catalogue actually stocks. That is the number a library designer sorts on.

The catalogue below is the reference's own cenobamate reagents plus a handful of azoles.

In [15]:
from synplan.chem.synthon.stock import SynthonRecord, write_synthon_stock

CATALOGUE = {
    "ClC(Cl)(Cl)C(=O)N=C=O": "trichloroacetyl isocyanate",
    "OC(CBr)C1=CC=CC=C1Cl": "2-bromo-1-(2-chlorophenyl)ethanol",
    "C1=NN=NN1": "1H-tetrazole",
    "CC1=NN=NN1": "5-methyltetrazole",
    "NC1=NN=NN1": "5-aminotetrazole",
    "C1=NC=NN1": "1,2,4-triazole",
    "C1=CNN=C1": "pyrazole",
    "C1=CN=CN1": "imidazole",
}

catalogue_records = [
    SynthonRecord(synthon, (smi,), tuple(sorted(record["classes"])), record["component"])
    for smi in CATALOGUE
    for synthon, record in synthoniser.synthonise_smiles(smi).items()
]
catalogue_path = results_folder / "cenobamate_synthons.smi"
write_synthon_stock(str(catalogue_path), catalogue_records)

catalogue = load_synthon_stock(str(catalogue_path))
print(f"{len(CATALOGUE)} building blocks -> {len(catalogue)} synthons")

8 building blocks -> 16 synthons


In [16]:
scored = Fragmenter(SynthonConfig(), catalogue).fragment(target)

for pathway in scored.best_available():
    print(f"{pathway.availability:5.3f}  {'|'.join(pathway.rules):28s} {' . '.join(pathway.key)}")

0.722  R2.2_0|R5.1_0                [CH_elec](=O)N . [nH]1cnn[n_nuc]1 . c1cccc(c1Cl)C([OH_nuc])[CH3_elec]
0.722  R2.2_0|R5.1_0|R16.2b_0       [CH2_elec]=[NH_nuc] . [CH_elec](=O)N . [NH_elec]=N[NH2_nuc] . c1cccc(c1Cl)C([OH_nuc])[CH3_elec]
0.167  R2.2_0                       [CH_elec](=O)N . c1(ccccc1C(Cn2nncn2)[OH_nuc])Cl
0.167  R2.2_0|R5.2_0                [CH_elec](=O)N . [OH_nuc]C(c1c(cccc1)Cl)[CH3_elecB] . [nH]1cnn[n_nuc]1
0.167  R2.2_0|R10.1_0               [CH_elec](=O)N . c1c(Cl)c([CH2_elec][OH_nuc])ccc1 . n1(ncnn1)[CH3_nuc]
0.167  R2.2_0|R10.1_1               [CH_elec](=O)N . [OH_nuc][CH2_elec]Cn1nncn1 . c1cccc(Cl)[cH_nuc]1
0.167  R2.2_0|R5.2_0|R16.2b_0       [CH2_elec]=[NH_nuc] . [CH_elec](=O)N . [NH_elec]=N[NH2_nuc] . [OH_nuc]C(c1c(cccc1)Cl)[CH3_elecB]
0.000  R5.1_0                       [CH3_elec]C(OC(N)=O)c1ccccc1Cl . [nH]1cnn[n_nuc]1
0.000  R5.2_0                       [nH]1cnn[n_nuc]1 . c1c(c(ccc1)C(OC(N)=O)[CH3_elecB])Cl
0.000  R5.1_0|R16.2b_0              [CH2_elec]=[NH_

The two-stage carbamoylation/alkylation pathway comes out on top at 0.72, the value the reference reports for the same molecule against its full Enamine catalogue. `availability_denominator` switches the denominator between the target's atoms and the pathway's own.

In [17]:
best = scored.best_available()[0]
print("|".join(best.rules), "->", best.availability)
for synthon in best.key:
    suppliers = catalogue.get(synthon)
    print(f"  {synthon:40s} {'from ' + ', '.join(CATALOGUE[b] for b in suppliers) if suppliers else 'NOT in stock'}")

R2.2_0|R5.1_0 -> 0.7222222222222222
  [CH_elec](=O)N                           from trichloroacetyl isocyanate
  [nH]1cnn[n_nuc]1                         NOT in stock
  c1cccc(c1Cl)C([OH_nuc])[CH3_elec]        from 2-bromo-1-(2-chlorophenyl)ethanol


## 7. Enumeration

Two modes, both driven by one join. The 29-row compatibility table says which labels may bond, and `join` draws that bond and consumes both attachment points; the reference spells this out as 38 reconstruction SMIRKS, every one of which is two reactants giving one product with exactly one new bond.

### Mode 1 — the full combinatorial library

`enumerate_library` grows a molecule from each synthon in turn until it has no open labels left. Given the 18 synthons from the nine reference building blocks, it reproduces all 47 products the reference publishes for that tutorial.

In [18]:
from synplan.chem.synthon.enumerate import Enumerator

wide = SynthonConfig(mw_lower=0.0, mw_upper=10_000.0, max_products=100_000)
library = {str(m) for m in Enumerator(wide).enumerate_library(sorted(tutorial_stock))}

published = {
    str(safe_canonicalization(smiles(line.strip())))
    for line in (data_folder / "final_result.smi").read_text().splitlines()
    if line.strip()
}
print(f"{len(library)} products enumerated")
print(f"{len(published)} published; all reproduced: {published <= library}")

show(sorted(library)[:6])

86 products enumerated
47 published; all reproduced: True


`mw_lower` / `mw_upper` bound the products (100-1000 Da by default), `max_reacted_synthons` bounds how many synthons a product may consume, and `max_products` bounds the output — the enumerator streams, so a capped run stops working rather than filtering a finished list.

The table here carries five rows the published one does not. Two are the aryl radical partners: a stocked aryl-BF3, MIDA boronate or aryl sulfinate synthon is emitted by the classifier but has no upstream row, so it used to raise. The other three are the Suzuki pairing, `elec` with `elecB`, read off the R12.1/R12.2/R12.6 reconstruction SMIRKS. Upstream the table is only a whole-molecule pre-filter and the SMIRKS form the bond; here the table *is* the join, so without those rows a biaryl disconnects and then reassembles to nothing.

In [19]:
rows = [tuple(row[:3]) + tuple(row[3:]) for row in load_data(SynthonConfig().rules_path)["pairs"]]

suzuki = [r for r in rows if "elecB" in r and "elec" in r]
radical = [r for r in rows if ("C", True, "nuc*") in (r[:3], r[3:])]

print(f"{len(rows)} rows; {len(rows) - len(suzuki) - len(radical)} of them the published table\n")
for what, added in (("Suzuki", suzuki), ("aryl radical", radical)):
    for row in added:
        print(f"  {what:12s} {row[:3]} <-> {row[3:]}")

29 rows; 24 of them the published table

  Suzuki       ('C', False, 'elec') <-> ('C', False, 'elecB')
  Suzuki       ('C', False, 'elecB') <-> ('C', True, 'elec')
  Suzuki       ('C', True, 'elec') <-> ('C', True, 'elecB')
  aryl radical ('C', False, 'elec*') <-> ('C', True, 'nuc*')
  aryl radical ('C', True, 'elec*') <-> ('C', True, 'nuc*')


### Mode 2 — analogues of one target

The other mode starts from a fragmentation pathway instead of a bare synthon pool: each slot is filled exactly once, from its own list of candidates. `SynthonStock.slots` builds those lists — the stocked synthon itself, plus its positional analogues when `find_analogues` is on, minus whatever the rule of two rejects.

This is the step that turns the paper's Library1 into Library2. The tetrazole slot of the best cenobamate pathway is *not* in the catalogue as cut, so with analogues off the slot is empty; with them on it is filled by the four azoles the reference finds for the same synthon.

In [20]:
config = SynthonConfig(find_analogues=True)
slots = catalogue.slots(best.key, config)

for synthon, candidates in slots.items():
    print(f"{synthon}\n    {candidates}")

[CH_elec](=O)N
    ['[CH_elec](=O)N']
[nH]1cnn[n_nuc]1
    ['n1nnc[nH_nuc]1', 'n1c([nH_nuc]nn1)C', 'n1c(N)[nH_nuc]nn1', 'c1nnc[nH_nuc]1']
c1cccc(c1Cl)C([OH_nuc])[CH3_elec]
    ['c1cccc(c1Cl)C([OH_nuc])[CH3_elec]']


In [21]:
analogues = list(Enumerator(config).enumerate_analogues(best.key, slots))
print(f"{len(analogues)} analogues of cenobamate")
show(analogues)

4 analogues of cenobamate


`strict_availability` decides what an empty slot means: off (the default) the pathway's own synthon is used anyway, on it vetoes the whole pathway, so only fully synthesizable analogues come out.

In [22]:
strict = SynthonConfig(find_analogues=True, strict_availability=True)
bare = catalogue.slots(best.key, SynthonConfig())  # analogues off: the tetrazole slot is empty
print("slots without analogues:", {k: v for k, v in bare.items()})
print("strict enumeration     :", list(Enumerator(strict).enumerate_analogues(best.key, bare)))

slots without analogues: {'[CH_elec](=O)N': ['[CH_elec](=O)N'], '[nH]1cnn[n_nuc]1': [], 'c1cccc(c1Cl)C([OH_nuc])[CH3_elec]': ['c1cccc(c1Cl)C([OH_nuc])[CH3_elec]']}
strict enumeration     : []


## 8. Positional analogue scanning

That slot widening is positional analogue scanning. Two hard gates come first — the same multiset of labels, and the same heavy-neighbour degree at every labelled atom — and because both are exact equality they are a dictionary key rather than a scan, which is what makes PAS tractable over a whole catalogue. What survives must then have the same number of rings, be at most one heavy atom away, and differ by one of four shapes: an isomeric rearrangement, an aromatic C/N swap, or the gain or loss of a single C, N, O or F.

The degree gate is stricter than the paper's "same types of reaction centres": `[NH2_nuc]` has degree 1 and `[NH_nuc]` degree 2, and they are not interchangeable.

In [23]:
from synplan.chem.synthon.analogues import find_analogues, index_for_analogues, is_analogue

index = index_for_analogues(catalogue)
query = synthon_smiles("[nH]1cnn[n_nuc]1")
query.canonicalize()

print("query:", query)
for found in find_analogues(query, index):
    print(f"  {found:25s} {', '.join(CATALOGUE[b] for b in catalogue[found])}")

query: [nH]1cnn[n_nuc]1
  n1nnc[nH_nuc]1            1H-tetrazole
  n1c([nH_nuc]nn1)C         5-methyltetrazole
  n1c(N)[nH_nuc]nn1         5-aminotetrazole
  c1nnc[nH_nuc]1            1,2,4-triazole


In [24]:
# what the four shapes look like, one by one
shapes = [
    ("n1nnc[nH_nuc]1", "same atoms, moved"),
    ("c1nnc[nH_nuc]1", "aromatic C/N swap"),
    ("n1c([nH_nuc]nn1)C", "gains one C"),
    ("[n_nuc]1c[nH]cc1", "two swaps away"),
]

candidates = []
labels = ["query"]
for smi, why in shapes:
    candidate = synthon_smiles(smi)
    candidate.canonicalize()
    candidates.append(candidate)
    labels.append(f"{is_analogue(query, candidate)!s:5s} {why}")

show([query] + candidates, labels)

`similarity_threshold` adds a Tanimoto route into the same slot as a union with PAS, not a replacement: `-1` disables it and leaves the PAS-only floor. `ro2_filtration` restricts the candidates to what the rule of two calls reagent-like.

## 9. Ring closure

This is the port's main addition and the reference has no counterpart for it: 76 of the 154 shipped rules close a ring rather than cut an acyclic bond, and `close_ring` — a `join` without the merge, because the two labels are already in the same molecule — is what draws the second, intramolecular bond.

A ring synthon is an ordinary H-capped fragment carrying two labels. Joining the first pair merges the partner in, which leaves the second pair intramolecular and so beyond anything `join` can express.

In [25]:
ring_rules = [r for r in rules if r["ring"]]
print(f"{len(ring_rules)} ring-closure rules of {len(rules)}\n")
for rule in ring_rules[:5]:
    print(f"{rule['id']:10s} {rule['name']}")

76 ring-closure rules of 154

R16.1a     1,2,3-triazole, N1-substituted / azide + alkyne (CuAAC, RuAAC, SPAAC, thermal Huisgen)
R16.1b     1,2,3-triazole, N-unsubstituted (1H) / azide (NaN3, TMS-N3) + alkyne [n;h1] twin of R16.1a
R16.2a     tetrazole, 1,5-disubstituted / organic azide + nitrile
R16.2b     tetrazole, 5-substituted N-unsubstituted (1H) / NaN3 + nitrile, [n;h1] twin of R16.2a
R16.3a     pyrazole, N1-substituted / Knorr (substituted hydrazine + 1,3-dicarbonyl, enaminone or ynone)


One worked heterocyclisation: 5-phenyltetrazole. `R16.2b` cuts it into a benzaldimine and a triazene — two bonds, not one — and the enumerator puts the ring back.

In [26]:
PHENYLTETRAZOLE = "c1ccccc1-c1nnn[nH]1"

ring_dag = fragment_smiles(PHENYLTETRAZOLE)
for pathway in ring_dag.best_available():
    print(f"{'|'.join(pathway.rules):22s} {' . '.join(pathway.key)}")

# two labels on each fragment: one pair joins the two, the other closes the ring
heterocyclisation = next(p for p in ring_dag.pathways.values() if p.rules == ("R16.2b_0",))
show(heterocyclisation.key)

R12.1_0                [cH_elec]1[nH]nnn1 . c1c[cH_elecB]ccc1
R16.2b_0               [NH_elec]=N[NH2_nuc] . c1ccccc1[CH_elec]=[NH_nuc]
R12.1_0|R16.2b_0       [CH2_elec]=[NH_nuc] . [NH_elec]=N[NH2_nuc] . c1c[cH_elecB]ccc1


In [27]:
slots_here = {s: [s] for s in heterocyclisation.key}

rebuilt = list(Enumerator(SynthonConfig()).enumerate_analogues(heterocyclisation.key, slots_here))
print("target    :", ring_dag.target)
print("round trip:", ring_dag.target in {str(m) for m in rebuilt})
show(rebuilt)

target    : c1cc(ccc1)-c2nnn[nH]2
round trip: True


`ring_closure_sizes` is the knob: the sizes a heterocyclisation may close, defaulting to `(5, 6, 7)` — 5 and 6 cover every azole and azine, 7 the diazepines. Setting it to `()` turns ring closure off entirely and restores the acyclic-only behaviour exactly, both when cutting and when rebuilding.

In [28]:
acyclic_only = SynthonConfig(ring_closure_sizes=())

print("cenobamate, rings on :", len(fragment_smiles(CENOBAMATE).pathways), "pathways")
print("cenobamate, rings off:", len(Fragmenter(acyclic_only).fragment(target).pathways), "pathways")
print()
print("phenyltetrazole rebuilt with rings off:",
      list(Enumerator(acyclic_only).enumerate_analogues(heterocyclisation.key, slots_here)))

cenobamate, rings on : 11 pathways
cenobamate, rings off: 7 pathways

phenyltetrazole rebuilt with rings off: []


## Results

The results folder now holds the two synthon stocks built above:

In [29]:
for path in (stock_path, catalogue_path):
    print(f"{path.name:26s} {len(path.read_text().splitlines())} synthons")

tutorial_synthons.smi      18 synthons
cenobamate_synthons.smi    16 synthons


## Where to go next

Everything above has a command-line equivalent, all of them reading the same `SynthonConfig`:

| Command | What it does |
| --- | --- |
| `synplan bb_classifying` | assigns building-block classes over a file |
| `synplan bb_synthonizing` | turns a catalogue into a synthon stock |
| `synplan bb_scaffolds` | Bemis-Murcko scaffolds after protecting-group removal |
| `synplan synthon_fragment` | cuts targets into disconnection pathways |
| `synplan synthon_enumerate` | recombines stocked synthons into molecules |
| `synplan synthon_coverage` | filters out reactions the shipped disconnections already cover |

Each one has a Python mirror in `synplan.interfaces.synthon_commands` — `classify_file`, `synthonise_file` (used in section 3), `scaffolds_file`, `fragment_file`, `enumerate_file`, `coverage_file` — taking the same config object.

The defaults are written out in `configs/synthonisation.yaml`; pass it with `--config` and change one line rather than re-deriving the object in Python. Setting `write_audit_files: true` makes any of the five workflows that produce molecules — every command above except `synthon_coverage` — write an audited bundle beside its primary output: `fallback.smi`, `fallback.tsv`, `errors.tsv`, `summary.json` and `run.log`.

`synthon_coverage` is the bridge back into ordinary retrosynthesis: it asks whether an atom-mapped reaction is already the disconnection of a shipped synthon rule, which is how a training corpus is stripped of chemistry the curated rules already provide. The same disconnections can also steer an MCTS search directly — see the [retrosynthesis with synthon priority rules tutorial](18_Retrosynthesis_With_Synthon_Priority_Rules.ipynb), and the [priority rules tutorial](13_Priority_Rules.ipynb) for the general mechanism with a hand-written rule set.